# TorchVision

# Transforms

`torchvision.transforms.v2` provides transformations for image preprocessing and data augmentation.

```python
from torchvision.transforms import v2
```

> **PIL (Pillow)** can be used to read `.JPEG`, `.PNG`, and other image formats. `torchvision` transforms can then operate on PIL images, tensors, and other supported inputs.

---

# 1. Data Preprocessing

Data preprocessing prepares the images so they have the format and dimensions expected by the model.

These transformations are generally **deterministic**.

## 1.1 Compose

`Compose` allows multiple transformations to be applied sequentially.

```python
transforms = v2.Compose([
    v2.Resize((256, 256)),
    v2.CenterCrop((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])
```

The transformations are applied in order:

```text
Image
  ↓
Resize
  ↓
CenterCrop
  ↓
ToImage
  ↓
ToDtype
  ↓
Tensor
```

---

## 1.2 ToImage

Converts supported image inputs, such as PIL images or NumPy arrays, into a torchvision image tensor.

```python
to_image = v2.ToImage()

img_tensor = to_image(img)
```

The values are generally still in the original integer range, such as:

```text
0 → 255
```

It does **not** automatically convert the tensor to `float32` or normalize the values to `[0, 1]`.

---

## 1.3 Resize

Changes the spatial dimensions of the image.

```python
resize = v2.Resize(
    size=(256, 256) # SIZE = the new WEIGHT x WIDTH of the image
)

image_resized = resize(image)
```

> Important `Resize` can also be used as part of data augmentation when combined with random transformations such as `RandomResizedCrop`.

---

## 1.4 ToDtype

Changes the dtype of the image tensor and can optionally rescale its values.

```python
dtype = v2.ToDtype(
    dtype=torch.float32, # The new dtype of the matrix
    scale=True # Scale the values to 0-1
)

image_float = dtype(image)
```
```text
uint8:   0 → 255
             ↓
float32: 0.0 → 1.0
```

This is commonly done before feeding images into a neural network.

---

## 1.5 Normalize

Normalizes the values of the image tensor.

```python
normalize = v2.Normalize(
    mean=[0.5, 0.5, 0.5],
    std=[0.5, 0.5, 0.5]
)

image_normalized = normalize(image)
```

The `mean` and `std` are normally specified separately for each channel.

For an RGB image:

```text
mean = [R_mean, G_mean, B_mean]
std  = [R_std,  G_std,  B_std]
```

---

# 2. Data Augmentation

Data augmentation artificially creates variations of training images.

The objective is to make the model more robust and reduce overfitting.

> **This is used in data augmentation.**

---

# 2.1 Geometric Transformations

These transformations change the **position, orientation, scale, or geometry** of objects.

## Horizontal Flip

Flips the image horizontally, like a mirror.

```python
horizontal_flip = v2.RandomHorizontalFlip(
    p=0.5 #  The probabilitie to flip 
)
```

`p=0.5` means there is a 50% probability of applying the transformation.

> **This is used in data augmentation.**

---

## Vertical Flip

Flips the image vertically.

```python
vertical_flip = v2.RandomVerticalFlip(
    p=0.5 # The probabilitie to flip 
)
```

Useful only when vertical flipping makes sense for the problem.

> **This is used in data augmentation.**

---

## Center Crop

Crops the center of the image.

```python
center_crop = v2.CenterCrop(
    size=(224, 224) # The new size will be (HEIGHT, WIDTH)
)
```

The borders are discarded.

```text
Original image
      ↓
Center region
      ↓
224 × 224
```

> `CenterCrop` is usually used as preprocessing when deterministic, but it can also be part of an augmentation pipeline.

---

## Random Crop

Randomly selects a region of the image.

```python
random_crop = v2.RandomCrop(
    size=(224, 224) # The new size will be (HEIGHT, WIDTH)
)

```

Different accesses to the same image can produce different crops.

```text
Image
  ↓
Random region
  ↓
224 × 224
```

> **This is used in data augmentation.**

---

## Random Resized Crop

Randomly crops a region and then resizes it to the desired size.

```python
random_resized_crop = v2.RandomResizedCrop(
    size=224, # The new size will be (HEIGHT, WIDTH)
    scale=(0.6, 1.0), # The proportion of the image to maintain (60 - 100%), so the max loss of the matrix will be 40%
    ratio=(3/4, 4/3),
    antialias=True
)
```

> **This is used in data augmentation.**

---

## Rotation

Randomly rotates the image.

```python
random_rotation = v2.RandomRotation(
    degrees=20 # Turns the image 20 degrees
)
```

This means:

```text
-20° ← random angle → +20°
```

> **This is used in data augmentation.**


---

## Random Affine

Applies random affine transformations such as rotation, translation, scaling, and shearing.

```python
random_affine = v2.RandomAffine(
    degrees=20,
    translate=(0.1, 0.1),
    scale=(0.8, 1.2),
    shear=10
)
```

> **This is used in data augmentation.**

---

## Random Perspective

Randomly changes the perspective of the image.

```python
random_perspective = v2.RandomPerspective(
    distortion_scale=0.5,
    p=0.5
)
```

It simulates the object being viewed from different perspectives.

```text
Normal view
     ↓
Perspective transformation
     ↓
Different viewpoint
```

> **This is used in data augmentation.**

---

# 2.2 Appearance Transformations

These transformations change the **visual appearance** of the image rather than its geometry.

## Color Jitter

Randomly changes brightness, contrast, saturation, and hue.

```python
color_jitter = v2.ColorJitter(
    brightness=0.2, # The variation of the brightness (20%)
    contrast=0.2, # The variation of the contrast (20%)
    saturation=0.2, # The variation of the saturation (20%)
    hue=0.1 # The variation of the huea, the variotion of the colors (0.1)
)
```

> **This is used in data augmentation.**

Useful when lighting and color conditions can vary in the real world.

---

## Gaussian Blur

Applies Gaussian blur to the image.

```python
blur = v2.GaussianBlur(
    kernel_size=5,
    sigma=(0.1, 2.0)
)
```

It reduces fine details and edge sharpness.

> **This is used in data augmentation.**

---

## Grayscale

Converts the image to grayscale.

```python
grayscale = v2.Grayscale(
    num_output_channels=3
)
```

It removes color information while keeping the output with 3 channels when `num_output_channels=3`.

> Usually used as part of **data augmentation** when color should not be essential for the task.

---

# 2.3 Occlusion Transformations

These transformations hide parts of the image.

## Random Erasing

Randomly removes a rectangular region of the image.

```python
random_erasing = v2.RandomErasing(
    p=0.5,
    scale=(0.02, 0.2),
    ratio=(0.3, 3.3)
)
```

Conceptually:

```text
Original image

████████████
████████████
████████████
████████████

      ↓

Random Erasing

████████████
██████░░░░██
██████░░░░██
████████████
```

The model must learn to recognize the object even when part of it is missing.

> **This is used in data augmentation.**

---

# 3. Summary by Purpose

## Data Preprocessing

Used to prepare the image for the model.

- ToImage
- Resize
- ToDtype
- Normalize
- CenterCrop
- Compose


## Geometric Data Augmentation

Changes the geometry or position of the object.

- RandomHorizontalFlip
- RandomVerticalFlip
- sRandomCrop
- RandomResizedCrop
- RandomRotation
- RandomAffine
- RandomPerspective

> **These are used in data augmentation.**

## Appearance Data Augmentation

Changes how the image looks.

- ColorJitter
- GaussianBlur
- Gaussian Noise
- RandomGrayscale

> **These are used in data augmentation.**

## Occlusion Data Augmentation

Hides parts of the image.
- RandomErasing

> **This is used in data augmentation.**


# 4. Training vs Test

Data augmentation should normally be applied to the **training set**, not the test set.

For example:

```python
train_transforms = v2.Compose([
    v2.RandomResizedCrop(224),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean, std)
])
```

While the test set can use only deterministic transformations:

```python
test_transforms = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean, std)
])
```